<a href="https://colab.research.google.com/github/Smyles019/html-login-form-detector/blob/feat%2Fxgboost/02_xgboost_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Smyles019/html-login-form-detector.git

Cloning into 'html-login-form-detector'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 51 (delta 12), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 363.90 KiB | 2.01 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [9]:
import os
import shutil
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [16]:
# 1. Reset and recreate processed directory
processed_path = 'html-login-form-detector/data/processed'
if os.path.exists(processed_path):
    if os.path.isfile(processed_path):
        os.remove(processed_path)
    elif os.path.isdir(processed_path):
        shutil.rmtree(processed_path)
os.makedirs(processed_path, exist_ok=True)

# 2. Comprehensive label schema map
raw_path = 'html-login-form-detector/data/raw/Dataset 1/'
label_mapping = {
    'LOGIN_FORM_MALICIOUS': 'LOGIN_FORM',
    'TEST_LOGIN_FORM': 'LOGIN_FORM',
    'LOGIN_FORM': 'LOGIN_FORM',
    'TEST_NO_FORM': 'NO_FORM',
    'NO_FORM': 'NO_FORM'
}

def load_and_clean(file_form, file_no_form):
    df1 = pd.read_csv(raw_path + file_form)
    df2 = pd.read_csv(raw_path + file_no_form)

    # Combine & deduplicate signatures
    df = pd.concat([df1, df2], ignore_index=True).drop_duplicates(subset=['html_signature'])

    # Fill text gaps and map target labels
    df['html_signature'] = df['html_signature'].fillna('')
    df['clean_label'] = df['label'].map(label_mapping).fillna(df['label'])

    return df.dropna(subset=['clean_label']).reset_index(drop=True)

# 3. Clean and save output datasets
df_train = load_and_clean('train_login_form.csv', 'train_no_form.csv')
df_val = load_and_clean('validation_login_form.csv', 'validation_no_form.csv')

df_train.to_csv(f'{processed_path}/train_cleaned.csv', index=False)
df_val.to_csv(f'{processed_path}/validation_cleaned.csv', index=False)

print("Unique labels in Train:", df_train['clean_label'].unique())
print("Unique labels in Val:  ", df_val['clean_label'].unique())
print(f"Data cleaned & saved! Train: {len(df_train)} rows | Val: {len(df_val)} rows")

Unique labels in Train: ['LOGIN_FORM' 'NO_FORM']
Unique labels in Val:   ['LOGIN_FORM' 'NO_FORM']
Data cleaned & saved! Train: 1141 rows | Val: 205 rows


In [24]:
# 1. Load cleaned datasets
df_train = pd.read_csv('html-login-form-detector/data/processed/train_cleaned.csv')
df_val = pd.read_csv('html-login-form-detector/data/processed/validation_cleaned.csv')

df_train['html_signature'] = df_train['html_signature'].fillna('')
df_val['html_signature'] = df_val['html_signature'].fillna('')

# 2. Encode target labels
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(df_train['clean_label'].astype(str))
y_val = label_encoder.transform(df_val['clean_label'].astype(str))

# 3. Vectorize text n-grams
vectorizer = CountVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=10000
)
X_train_text = vectorizer.fit_transform(df_train['html_signature'])
X_val_text = vectorizer.transform(df_val['html_signature'])

X_train = X_train_text
X_val = X_val_text

# 4. Auto-detect available numeric features safely
expected_numeric = ['max_depth', 'total_tags', 'form_count', 'input_count', 'button_count', 'script_count']
available_numeric = [col for col in expected_numeric if col in df_train.columns]

if available_numeric:
    print(f"Scaling numeric features: {available_numeric}")
    scaler = RobustScaler()
    X_train_num = scaler.fit_transform(df_train[available_numeric])
    X_val_num = scaler.transform(df_val[available_numeric])
    X_train = hstack([X_train_num, X_train_text])
    X_val = hstack([X_val_num, X_val_text])
else:
    print("No numeric DOM columns found. Proceeding with text n-grams only!")
    X_train = X_train_text
    X_val = X_val_text

print("Classes mapped:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)

No numeric DOM columns found. Proceeding with text n-grams only!
Classes mapped: {'LOGIN_FORM': np.int64(0), 'NO_FORM': np.int64(1)}
X_train shape: (1141, 10000)
X_val shape:   (205, 10000)


In [25]:
# Initialize and fit XGBoost
model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    scale_pos_weight=1.5,       # Gives extra importance to LOGIN_FORM
    early_stopping_rounds=20,
    random_state=42,
    eval_metric='logloss'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=25
)

[0]	validation_0-logloss:0.74451
[25]	validation_0-logloss:0.59529
[50]	validation_0-logloss:0.51873
[75]	validation_0-logloss:0.47415
[100]	validation_0-logloss:0.45500
[125]	validation_0-logloss:0.45349
[127]	validation_0-logloss:0.45426


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=20,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [26]:
y_proba = model.predict_proba(X_val)[:, 0]  # Probability of LOGIN_FORM (Class 0)

# Lower threshold to pick up missed forms (e.g., 0.35 instead of 0.50)
custom_threshold = 0.35
y_pred_adjusted = np.where(y_proba >= custom_threshold, 0, 1)

print(f"=== Adjusted Evaluation (LOGIN_FORM Threshold = {custom_threshold}) ===")
print(classification_report(y_val, y_pred_adjusted, target_names=label_encoder.classes_))
print(f"ROC-AUC Score: {roc_auc_score(y_val, y_proba):.4f}")

# Updated Confusion Matrix
cm = confusion_matrix(y_val, y_pred_adjusted)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {c}" for c in label_encoder.classes_],
    columns=[f"Pred {c}" for c in label_encoder.classes_]
)
print("\n=== Confusion Matrix ===")
print(cm_df)

=== Adjusted Evaluation (LOGIN_FORM Threshold = 0.35) ===
              precision    recall  f1-score   support

  LOGIN_FORM       0.96      0.67      0.79       103
     NO_FORM       0.74      0.97      0.84       102

    accuracy                           0.82       205
   macro avg       0.85      0.82      0.82       205
weighted avg       0.85      0.82      0.82       205

ROC-AUC Score: 0.0618

=== Confusion Matrix ===
                   Pred LOGIN_FORM  Pred NO_FORM
Actual LOGIN_FORM               69            34
Actual NO_FORM                   3            99


In [22]:
import joblib

# Save model, vectorizer, and label encoder
os.makedirs('html-login-form-detector/models', exist_ok=True)
joblib.dump(model, 'html-login-form-detector/models/xgboost_model.pkl')
joblib.dump(vectorizer, 'html-login-form-detector/models/vectorizer.pkl')
joblib.dump(label_encoder, 'html-login-form-detector/models/label_encoder.pkl')

print("Model pipeline successfully saved to 'html-login-form-detector/models/'!")

Model pipeline successfully saved to 'html-login-form-detector/models/'!
